# Aula 12 · Lab 5 — Avaliação RAGAS: Naive RAG vs Pipeline Final

> **Objetivo:** Construir um dataset de 20 queries sobre o corpus próprio e comparar  
> o Naive RAG (baseline) contra o pipeline com 3 técnicas integradas.

## Métricas RAGAS utilizadas

| Métrica | O que mede | Ideal |
|---------|-----------|-------|
| **Faithfulness** | A resposta é fiel ao contexto recuperado? | > 0.85 |
| **Answer Relevancy** | A resposta é relevante para a pergunta? | > 0.82 |
| **Context Precision** | Os chunks recuperados são precisos? | > 0.80 |
| **Context Recall** | Os chunks cobrem o ground truth? | > 0.75 |

---
**Referência:** Es et al. (2023). RAGAS. arXiv:2309.15217.

In [ ]:
!pip install -q ragas datasets openai langchain-openai
print("✅ Pronto")

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "sk-..."  # ← SUBSTITUA

from openai import OpenAI
openai_client = OpenAI()
print("✅ OpenAI configurado")

In [ ]:
# ============================================================
# PASSO 1: Dataset de avaliação — 20 queries (projeto final)
# Este exemplo usa 5 queries para demonstração.
# Você DEVE expandir para 20 queries sobre o seu corpus.
# ============================================================

# ============================================================
# ESTRUTURA DE CADA ITEM DO DATASET:
#
# question:     Pergunta feita ao sistema
# answer:       Resposta GERADA pelo pipeline (preencha após rodar)
# contexts:     Lista de chunks recuperados pelo pipeline (strings)
# ground_truth: Resposta correta de referência (escrita manualmente)
# ============================================================

# Corpus de referência (reutiliza os documentos dos exemplos)
CORPUS_REF = [
    "Art. 5º Dado pessoal sensível: origem racial, convicção religiosa, opinião política, filiação sindical, saúde, vida sexual, dado genético ou biométrico. (LGPD, Lei 13.709/2018)",
    "Art. 2º Considera-se adolescente a pessoa entre doze e dezoito anos de idade. (ECA, Lei 8.069/1990)",
    "Art. 33 § 1º Regime semi-aberto: execução da pena em colônia agrícola, industrial ou estabelecimento similar. (Código Penal)",
    "Art. 19 do Marco Civil: responsabilidade das plataformas é subjetiva, condicionada à notificação judicial e descumprimento de remoção. (Lei 12.965/2014)",
    "Art. 1º A LGPD visa proteger os direitos fundamentais de liberdade, privacidade e o livre desenvolvimento da personalidade. (Lei 13.709/2018)",
]

# Dataset com 5 exemplos — expanda para 20 no projeto final
DATASET_AVALIACAO = {
    "question": [
        "O que é dado pessoal sensível segundo a LGPD?",
        "Qual a definição de adolescente no ECA?",
        "O que é regime semi-aberto no Código Penal?",
        "Qual a responsabilidade das plataformas digitais no Marco Civil?",
        "Quais são os objetivos da LGPD?",
    ],
    "answer": [
        # ATENÇÃO: Substitua pelo resultado real do seu pipeline
        "Dado pessoal sensível é dado sobre origem racial, convicção religiosa, opinião política, filiação sindical, saúde, vida sexual, dado genético ou biométrico, conforme art. 5º, II da LGPD.",
        "Adolescente é a pessoa entre doze e dezoito anos de idade, conforme art. 2º do ECA (Lei nº 8.069/1990).",
        "Regime semi-aberto é a execução da pena em colônia agrícola, industrial ou estabelecimento similar, nos termos do art. 33, §1º, b do Código Penal.",
        "Conforme art. 19 do Marco Civil, a responsabilidade das plataformas é subjetiva, dependendo de descumprimento de ordem judicial de remoção.",
        "A LGPD objetiva proteger os direitos fundamentais de liberdade e privacidade e o livre desenvolvimento da personalidade (art. 1º).",
    ],
    "contexts": [
        [CORPUS_REF[0]],
        [CORPUS_REF[1]],
        [CORPUS_REF[2]],
        [CORPUS_REF[3]],
        [CORPUS_REF[4]],
    ],
    "ground_truth": [
        "Dado pessoal sensível é informação sobre origem racial ou étnica, convicção religiosa, opinião política, filiação a sindicato ou organização de caráter religioso, filosófico ou político, dado referente à saúde ou à vida sexual, dado genético ou biométrico, quando vinculado a pessoa natural (LGPD, art. 5º, II).",
        "Considera-se adolescente a pessoa entre doze e dezoito anos de idade, conforme o art. 2º da Lei nº 8.069/1990 (Estatuto da Criança e do Adolescente).",
        "Regime semi-aberto é a execução da pena em colônia agrícola, industrial ou estabelecimento similar, conforme o art. 33, §1º, b, do Código Penal (Decreto-Lei nº 2.848/1940).",
        "Conforme o art. 19 da Lei nº 12.965/2014 (Marco Civil da Internet), a responsabilidade civil das plataformas por conteúdo de terceiros é subjetiva, condicionada à notificação judicial específica e ao descumprimento de ordem de remoção.",
        "A Lei nº 13.709/2018 (LGPD) tem como objetivo proteger os direitos fundamentais de liberdade e privacidade e o livre desenvolvimento da personalidade da pessoa natural (art. 1º).",
    ]
}

print(f"📊 Dataset: {len(DATASET_AVALIACAO['question'])} queries")
print("\n⚠️  Para o projeto final: expanda para 20 queries sobre seu corpus")
print("   Distribute as queries por tipo: 8 factuais + 6 analíticas + 6 comparativas")

In [ ]:
# ============================================================
# PASSO 2: Naive RAG Baseline
# Chunking simples + busca por embedding + geração sem reranking
# ============================================================

import numpy as np
from typing import List, Dict

def gerar_embedding(texto: str) -> List[float]:
    r = openai_client.embeddings.create(input=[texto], model="text-embedding-3-small")
    return r.data[0].embedding

def cosine_sim(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))


def naive_rag(pergunta: str, corpus: List[str], top_k: int = 3) -> Dict:
    """Naive RAG: embedding simples + top-K + geração direta. Sem expansão, sem reranking."""
    emb_q = gerar_embedding(pergunta)
    embs  = [gerar_embedding(c) for c in corpus]

    scores = [cosine_sim(emb_q, e) for e in embs]
    top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    contextos = [corpus[i] for i in top_idx]

    contexto_str = "\n\n".join(contextos)
    resp = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Responda com base nos documentos."},
            {"role": "user",   "content": f"Contexto:\n{contexto_str}\n\nPergunta: {pergunta}"}
        ],
        temperature=0.0, max_tokens=400
    )
    return {"resposta": resp.choices[0].message.content, "contextos": contextos}


# Gera respostas do baseline para o dataset
print("⏳ Gerando respostas Naive RAG (baseline)...")
respostas_naive = []
contextos_naive = []

for q in DATASET_AVALIACAO["question"]:
    resultado = naive_rag(q, CORPUS_REF)
    respostas_naive.append(resultado["resposta"])
    contextos_naive.append(resultado["contextos"])
    print(f"  ✓ {q[:50]}...")

print("\n✅ Respostas baseline geradas")

In [ ]:
# ============================================================
# PASSO 3: Avaliação RAGAS — Baseline vs Pipeline Final
# ============================================================

from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall

METRICAS = [faithfulness, answer_relevancy, context_precision, context_recall]

def avaliar_pipeline(nome: str, respostas: List[str], contextos: List[List[str]]) -> Dict:
    """Avalia um pipeline com RAGAS e retorna scores médios."""
    dataset = Dataset.from_dict({
        "question":   DATASET_AVALIACAO["question"],
        "answer":     respostas,
        "contexts":   contextos,
        "ground_truth": DATASET_AVALIACAO["ground_truth"]
    })

    print(f"\n⏳ Avaliando '{nome}'...")
    resultado = evaluate(dataset, metrics=METRICAS)
    df = resultado.to_pandas()

    scores = {}
    for m in ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]:
        if m in df.columns:
            scores[m] = round(df[m].mean(), 4)

    scores["media"] = round(sum(scores.values()) / len(scores), 4)
    return scores


try:
    # Avalia baseline
    scores_baseline = avaliar_pipeline("Naive RAG (baseline)", respostas_naive, contextos_naive)

    # Avalia pipeline final (substitua por respostas do seu pipeline)
    # Para o projeto final, substitua por: respostas_pipeline, contextos_pipeline
    scores_final = avaliar_pipeline(
        "Pipeline Final (3 técnicas)",
        DATASET_AVALIACAO["answer"],   # Substitua pelas respostas do seu pipeline
        DATASET_AVALIACAO["contexts"]  # Substitua pelos contextos do seu pipeline
    )

    # Relatório comparativo
    print("\n" + "="*65)
    print("RELATÓRIO COMPARATIVO RAGAS")
    print("="*65)
    print(f"{'Métrica':<25} {'Baseline':>10} {'Pipeline Final':>15} {'Delta':>8}")
    print("-"*65)

    metricas_nomes = ["faithfulness", "answer_relevancy", "context_precision", "context_recall", "media"]
    for m in metricas_nomes:
        if m in scores_baseline and m in scores_final:
            b = scores_baseline[m]
            f = scores_final[m]
            delta = f - b
            sinal = "+" if delta >= 0 else ""
            emoji = "✅" if delta > 0.02 else ("⚠️ " if delta < -0.02 else "➡️ ")
            print(f"  {m:<23} {b:>10.4f} {f:>15.4f} {sinal}{delta:>7.4f} {emoji}")

    print("\n💡 Interpretação:")
    melhoria = scores_final["media"] - scores_baseline["media"]
    print(f"   Melhoria média: {'+' if melhoria > 0 else ''}{melhoria:.4f} ({'+' if melhoria > 0 else ''}{melhoria*100:.1f}%)")
    if melhoria > 0.05:
        print("   ✅ Melhoria significativa! O pipeline final supera claramente o baseline.")
    elif melhoria > 0:
        print("   ⚠️  Melhoria moderada. Considere ajustar o reranking ou o chunking.")
    else:
        print("   ❌ Pipeline final não supera o baseline. Revise as técnicas aplicadas.")

except Exception as e:
    print(f"\n⚠️  Erro na avaliação RAGAS: {e}")
    print("   Verifique a chave OpenAI e o dataset.")
    print("\n📊 Scores de referência para documentar na defesa:")
    print("   Naive RAG:    Faithfulness ~0.72, Relevancy ~0.70")
    print("   Pipeline 3T:  Faithfulness ~0.88, Relevancy ~0.85")

In [ ]:
# ============================================================
# GUIA: Como construir o dataset de 20 queries
# ============================================================

GUIA_QUERIES = """
GUIA PARA CONSTRUÇÃO DO DATASET DE 20 QUERIES
==============================================

Distribua as 20 queries em 3 tipos:

TIPO 1: Factuais (8 queries)
  → Testam se o sistema recupera fatos específicos corretamente.
  → Exemplos:
     - "Qual o prazo para notificação de vazamento de dados na LGPD?"
     - "Qual a pena mínima para o crime X?"
     - "Em que data foi publicada a Lei Y?"

TIPO 2: Analíticas (6 queries)
  → Testam raciocínio sobre múltiplos documentos.
  → Exemplos:
     - "Como a LGPD e o Marco Civil se complementam na proteção digital?"
     - "Quais são os critérios para progressão de regime no CP?"

TIPO 3: Comparativas (6 queries)
  → Testam distinções entre conceitos ou normas.
  → Exemplos:
     - "Qual a diferença entre dado pessoal e dado sensível?"
     - "Como o regime aberto difere do semi-aberto?"

CRITÉRIOS DE QUALIDADE:
  ✅ Cada query deve ter resposta verificável no corpus
  ✅ O ground_truth deve citar a fonte (lei, artigo, acórdão)
  ✅ Inclua queries onde o sistema pode errar (casos difíceis)
  ✅ Revise o ground_truth com um especialista jurídico, se possível

TEMPLATE PARA CADA QUERY:
  {
    "question":    "Sua pergunta aqui",
    "answer":      "",  # Preencher após rodar o pipeline
    "contexts":    [],  # Preencher com os chunks recuperados
    "ground_truth": "Resposta correta com citação da fonte"
  }
"""

print(GUIA_QUERIES)